In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
os.environ["OLLAMA_MODEL"] = os.getenv("OLLAMA_MODEL")

In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=os.environ["OLLAMA_MODEL"])

response = model.invoke("How are you doing today?")

print(response.content)

I'm doing well, thank you for asking! As an AI, I don't really "feel" in the way humans do, but I am fully functional and ready to assist you with anything you need.

How about yourself? How are *you* doing today? 😊


In [4]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model=os.environ["OLLAMA_MODEL"].split(":")[-1]
)

response = model.invoke("How are you doing today?")

print(response.content)

I'm doing very well, thank you for asking! As an AI, I don't have feelings, but I am fully operational and ready to assist you with any questions or tasks you have.

How are *you* doing today?


# Streaming

In [5]:
for chunk in model.stream("Write me a poem about the moon.", stream=True):
    print(chunk.content, end="", flush=True)

***The Watcher's Orb***

A polished coin against the velvet night,
You hang serene, an echo of pure light.
Oh silent keeper in the endless black,
The faithful orb upon the stellar track.

When you are sliver, sharp and newborn bright,
A spectral sickle stealing through the night;
You hint at fullness, promise soft and deep,
While all the dreaming, weary mortals sleep.

We mark your waxing to your waning slow,
The rhythmic tides that pull the currents flow.
From fragile Crescent to a perfect round,
Your silent magic on the waters found;
The gravitational whisper, steady hand,
That guides the voyage across every strand.

Sometimes you blaze in glory, full and wide,
A mystic mirror where all secrets ride.
You draw the dreamer up into your gaze,
And illuminate the night’s secret maze.

You watch us turning through our passing years,
Through whispered hopes and silent, falling tears.
The lonely watcher, timeless, pale, and grand,
A quiet mystery across the darkened land.

Oh Moon, you are t

# Batch

In [6]:
responses = model.batch([
    "Write me a poem about the sun.",
    "Write me a poem about the stars.",
    "Write me a poem about the clouds."
])

for response in responses:
    print(response.content)

**The Golden Architect**

A boundless disc that rides the velvet blue,
It wakes the sleeping world with liquid hue.
Before the curtain lifts and shadows flee,
You spill your saffron fire across the sea;
A silent promise whispered through the east,
The gentle wash of dawn’s first golden feast.

Oh, fierce celestial forge, magnificent crown,
You hoist yourself above the sleeping town.
You are the source, the pulse that keeps us bright,
Translating ancient darkness into light.
From whisper-glow to zenith's blazing might,
You flood the forest and you kiss the height.

The mountain peak is drenched in copper gleam,
Each dewdrop mirrors your ecstatic beam.
Through tangled canopy or open plain,
Your steady radiance washes out all pain.
You wake the flower with a warming grace,
And draw deep sustenance from time and space;
Giving life to roots that pierce the clay,
Energizing every passing day.

When clouds drift by like sails of endless white,
We feel your heat, both glorious and bright.
A so

In [7]:
query = [
    "Write me a poem about the sun.",
    "Write me a poem about the stars.",
    "Write me a poem about the clouds."
]

for response in model.batch_as_completed(query):
    print(response[1].content)


A wash of indigo, where darkness lies profound,
And silence reigns in spheres of stellar ground.
You hang above us, jewels upon the cloak of night,
Ancient beacons burning with eternal light.

From whispers whispered through a cosmic haze,
To blazing clusters of forgotten days,
Each pinprick dot, a sun of fiercer fire,
Fulfilling nebulae's deep desire.

The Hunter’s belt, a perfect, jeweled line,
Or constellations elegantly divine;
We trace the shapes that myths have made for sport—
The Dipper ladle, or the mighty sword report.
They guide the sailor through the boundless stream,
A tapestry woven from an endless dream.

Oh, the distance stretched across your silver spray,
In silent years that mark another day!
You burn with echoes of a primal gleam,
A timeless fire in a cosmic beam.
You carry light that left its source so long ago,
Whispering secrets only starlight knows.

When mortal worries dim beneath your might,
And fears dissolve before the stellar sight,
We feel ourselves so wonde

# Tools

### 1. Tools extend what agents can do—letting them fetch real-time data, execute code, query external databases, and take actions in the world.
### 2. Under the hood, tools are callable functions with well-defined inputs and outputs that get passed to a chat model. The model decides when to invoke a tool based on the conversation context, and what input arguments to provide.

In [8]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model=os.environ["OLLAMA_MODEL"])

In [9]:
from langchain.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """ Fetches the current weather for a given location.
        Args:
            location (str): The location for which to fetch the weather.
    Returns:
        str: A string describing the current weather in the specified location.
    """
    return f"The current weather in {location} is sunny."

model_with_tool = model.bind_tools([get_current_weather])

In [10]:
response = model_with_tool.invoke("What is the current weather in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']} with args: {tool_call['args']}")
    #print(f"Tool output: {tool_call['output']}")

content='' additional_kwargs={} response_metadata={'model': 'Gemma4', 'created_at': '2026-08-08T09:34:26.53773Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1830525125, 'load_duration': 252148583, 'prompt_eval_count': 111, 'prompt_eval_duration': 792426000, 'eval_count': 18, 'eval_duration': 780138000, 'logprobs': None, 'model_name': 'Gemma4', 'model_provider': 'ollama'} id='lc_run--019fe0b9-3fc2-7e11-a355-75b2c6fbf16f-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': '5f8ece19-8269-4b24-880d-7751e2191d55', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 111, 'output_tokens': 18, 'total_tokens': 129}
Tool called: get_current_weather with args: {'location': 'New York'}


In [11]:
# Step 1: Model generates tool calls based on the input query.
messages = [
    {"role": "user", "content": "What is the current weather in New York?"}
]
ai_message = model_with_tool.invoke(messages)
messages.append(ai_message)

# Step 2: Execute the tool calls generated by the model.
for tool_call in ai_message.tool_calls:
    tool_result = get_current_weather.invoke(tool_call)
    # Step 3: Append the tool results to the conversation.
    messages.append(tool_result)

# Step 4: Model generates a final response based on the tool results.
final_response = model_with_tool.invoke(messages)
print(final_response.content)

The current weather in New York is sunny.


In [12]:
messages

[{'role': 'user', 'content': 'What is the current weather in New York?'},
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'Gemma4', 'created_at': '2026-08-08T09:34:36.262657Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9711265458, 'load_duration': 249658291, 'prompt_eval_count': 111, 'prompt_eval_duration': 164314000, 'eval_count': 187, 'eval_duration': 9295954000, 'logprobs': None, 'model_name': 'Gemma4', 'model_provider': 'ollama'}, id='lc_run--019fe0b9-46f5-72f2-bdc3-21f98b016a96-0', tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': '7cf5bd5b-132c-499c-9f4d-39a5e4919bb1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 187, 'total_tokens': 298}),
 ToolMessage(content='The current weather in New York is sunny.', name='get_current_weather', tool_call_id='7cf5bd5b-132c-499c-9f4d-39a5e4919bb1')]

# Messages

### Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

### Messages are objects that contain:
### Role - Identifies the message type (e.g. system, user)
### Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
### Metadata - Optional fields such as response information, message IDs, and token usage

In [13]:
model.invoke("Who is the president of England?")

AIMessage(content='The concept of an "President of England" is technically inaccurate because the governance structure of England is part of the United Kingdom, which is a constitutional monarchy.\n\nTherefore, there isn\'t one person who holds the title of President for England. Instead, power and leadership are split among different roles:\n\n### 👑 Head of State\nThe ceremonial **Head of State** for England (and the entire UK) is **the Monarch**, currently King Charles III. He performs constitutional duties but does not lead the government day-to-day.\n\n### 🏛️ Head of Government\nThe person who exercises political and executive power, leads the government, and sets policy is the **Prime Minister**. The Prime Minister is responsible for the administration of England and the UK as a whole.\n\n***\n\n**In summary:**\n\n*   If you are asking for the person in charge of day-to-day governance and executive action: It is the **Prime Minister**.\n*   If you are asking who represents England

In [4]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem about the rain")
]

response = model.invoke(messages)

print(response.content)

***The Ode to Aqueous Veil***

The sky, a watercolour washed to pearl and slate,
Bows down its weight; an ancient sigh of fate.
It starts with silence—a mere suggestion soft,
A hesitant touch upon the sheltering loft.
Then comes the whisper, thin as silver thread,
A curtain drawn from heaven overhead.

It gathers strength, no longer meek and slow,
But finds a rhythmic pulse, a steady flow.
The drumming starts upon the roof's incline,
A hundred whispers in a sound divine.
It dances down the pane with staccato beat,
Washing the dust from life with rain-streaked feet.

Oh, the scent of it! The sweet petrichor release,
That rises up to promise perfect peace.
The thirsty earth breathes deep a damp content,
As dusty leaves of summer are relent.
The vibrant green of roots and struggling vine,
Awakens in this cleansing, liquid shine.

It speaks of sorrow and it sings of grace,
A quiet solace held within its space.
It settles down the fevered mind's harsh call,
And teaches hearts to yield when 

In [6]:
system_message = SystemMessage("You are a helpful coding assistant")
human_message = HumanMessage("How do i create a rest api?")

messages = [system_message, human_message]

res = model.invoke(messages)

print(res.content)

This is one of the biggest questions in modern software development! Creating a REST API involves far more than just writing code; it requires understanding design principles, architecture, and how data flows over HTTP.

Since "How do I create an API?" doesn't have a single answer (it depends heavily on your language choice), I will provide you with a comprehensive **five-stage roadmap**, moving from theory to execution, so you can understand the entire process regardless of whether you choose Python, JavaScript, Java, or another language.

---

## 🗺️ The 5-Stage Roadmap to Building a REST API

### Stage 1: Understand the Theory (The "Why")

Before writing any code, you must understand what you are building and why it works.

#### 🅰️ What is an API?
An **API (Application Programming Interface)** is simply a set of rules and protocols that allows different pieces of software to talk to each other. It acts as a messenger or intermediary. Your API defines *how* another application can ask

In [7]:
## Detailed info to the llm through the system message. The system message is used to set the behavior of the model. The human message is used to provide input to the model. The AI message is used to provide output from the model.

system_message = SystemMessage("""
You are a senior Python developer with expertise in building RESTful APIs. 
Your task is to provide clear and concise guidance on how to create a REST API using Python, including best practices, recommended frameworks, and code examples.
Be concise, informative, and provide step-by-step instructions where applicable.
""")

human_message = HumanMessage("How do I create a REST API using Python?")

messages = [system_message, human_message]

response = model.invoke(messages)

print(response.content)

As a senior developer, I recommend using **FastAPI** for new projects. While Flask is excellent for smaller microservices or rapid prototyping, FastAPI is built with modern standards in mind (type hinting, asynchronous programming) and automatically implements industry-standard best practices, significantly simplifying the creation of robust, high-performance APIs.

Here is a complete guide on creating your REST API using Python.

---

## 🚀 Recommended Framework: FastAPI

FastAPI provides automatic OpenAPI (Swagger UI/Redoc) documentation, built-in data validation (using Pydantic), and superior performance due to its foundation on Starlette and Uvicorn.

### Step 1: Setup and Installation

First, set up a virtual environment and install the necessary packages.

```bash
# Create and activate virtual environment
python -m venv venv
source venv/bin/activate  # On Linux/macOS
# .\venv\Scripts\activate # On Windows

# Install FastAPI (the framework) and Uvicorn (the ASGI server)
pip install